In [ ]:
import pandas as pd 
import numpy as np
import os
import matplotlib.pyplot as plt
import re

In [ ]:
import yaml 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import torch

In [ ]:
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from gensim.models import KeyedVectors
from gensim.parsing.preprocessing import remove_stopword_tokens

In [ ]:
from data_preprocess import create_data

In [ ]:
import torch 
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkLoader, LinkNeighborLoader, NeighborLoader

In [ ]:
config_file_path = 'config.yaml'

In [ ]:
users, movies, train_ratings, val_ratings, test_ratings, config = create_data(config_file_path)

In [ ]:
users

In [ ]:
movies

In [ ]:
train_ratings

In [ ]:
w2v_model = KeyedVectors.load_word2vec_format('data/GoogleNews-vectors-negative300_2.bin', binary=True)

In [ ]:
def compute_average_embedding(genres, w2v_model):
    embeddings = [w2v_model.get_vector(genre) for genre in genres if genre in w2v_model.index_to_key]
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        # Return a zero vector if no genres are found in the model
        return np.zeros(w2v_model.vector_size)
    
def preprocess_genre(genre):
    genre = genre.lower()
    genre = genre.split('|')
    return genre
    
def preprocess_title(title):
    title = title.lower()
    title = title.split(' ')
    title = remove_stopword_tokens(title)
    title  = [re.sub(r"[^ a-zA-Z0-9]+",'',word) for word in title]
    title = [word.strip() for word in title]
    title = [word for word in title if len(word)]
    return title
    

In [ ]:
preprocess_title('sd dsdbfDFjnv $%#$^#$ dskgjs,fs is the')

In [ ]:
def transform(users, movies):
    users['Gender'] = users['Gender'].map({'M' : 1, 'F' : 0})
    # Regex to extract title and year
    movies[['Title', 'Year']] = movies['Title'].str.extract(r'^(.*?)(?: \((\d{4})\))?$')
    movies['Year'].fillna(0, inplace = True)
    movies['Genre_List'] = movies['Genres'].apply(preprocess_genre)
    movies['Genre_Embedding'] = movies['Genre_List'].apply(lambda x: compute_average_embedding(x, w2v_model))
    movies['Title_List'] = movies['Title'].apply(preprocess_title)
    movies['Title_Embedding'] = movies['Title_List'].apply(lambda x: compute_average_embedding(x, w2v_model))
    return users, movies

In [ ]:
users, movies = transform(users, movies)

In [ ]:
movies

In [ ]:
users

In [ ]:
user_encoder = LabelEncoder()


In [ ]:
train_ratings['MovieID'] = train_ratings['MovieID'] - 1
train_ratings['UserID'] = train_ratings['UserID'] - 1

In [ ]:
train_pos_ratings = train_ratings[train_ratings['Rating'] >= 4]

In [ ]:
train_pos_ratings

In [ ]:
torch.tensor(users[['Gender', 'Age', 'Occupation']].values)

In [ ]:
# from torch_geometric.data import HeteroData

# data = HeteroData()

# # Create two node types "paper" and "author" holding a feature matrix:
# data['paper'].x = torch.randn(num_papers, num_paper_features)
# data['author'].x = torch.randn(num_authors, num_authors_features)

# # Create an edge type "(author, writes, paper)" and building the
# # graph connectivity:
# data['author', 'writes', 'paper'].edge_index = ...  # [2, num_edges]

# data['paper'].num_nodes
# >>> 23

# data['author', 'writes', 'paper'].num_edges
# >>> 52

# # PyTorch tensor functionality:
# data = data.pin_memory()
# data = data.to('cuda:0', non_blocking=True)

In [ ]:
data = HeteroData()

In [ ]:
data['movie'].x = torch.tensor(movies['Genre_Embedding'].apply(lambda x : x.tolist()) + movies['Title_Embedding'].apply(lambda x : x.tolist())+ movies['Year'].apply(lambda x: [int(x)]))

In [ ]:
data['users'].x = torch.tensor(users[['Gender', 'Age', 'Occupation']].values)

In [ ]:
data['movie']

In [ ]:
data['users']

In [ ]:
edge_index = train_pos_ratings[['UserID', 'MovieID']].values.T

In [ ]:
data['user', 'likes', 'movie'].edge_index = torch.tensor(edge_index)

In [ ]:
data

In [ ]:
data

In [ ]:
data['movie']

In [ ]:
data.edge_types

In [ ]:
data

In [ ]:
# loader = NeighborLoader(
#     data,
#     # Sample 30 neighbors for each node for 2 iterations
#     num_neighbors={key: [3] * 2 for key in data.edge_types},
#     # Use a batch size of 128 for sampling training nodes
#     batch_size=128,
#     input_nodes=('users', [1,2])
# )


In [ ]:
from graphviz import Digraph

# Create a directed graph for the ER diagram
dot = Digraph()

# Add nodes for the tables
dot.node('User', 'User Table\nuser_id (PK)\nage\ngender\noccupation\nzipcode')
dot.node('Movie', 'Movie Table\nmovie_id (PK)\ntitle\ngenre')
dot.node('Ratings', 'Ratings Table\nuser_id (FK)\nmovie_id (FK)\nrating\ntimestamp')

# Add edges to represent relationships
dot.edge('User', 'Ratings', label='1-to-many')
dot.edge('Movie', 'Ratings', label='1-to-many')

# Save the ER diagram as a PNG file
output_file = 'er_diagram_user_movie_ratings.png'
dot.render(output_file, format='png', cleanup=True)

output_file